# 🔬 Customer Health Forensics — Phase 2
## XAI Consensus Engine (Agreement-Based)

**Requires:** Phase 1 complete (best_model.pkl + feature_names.json in Drive)  
**Runtime:** Google Colab T4  
**Expected time:** ~5–10 min (200 watchlist explanations)

**What this phase does:**
- Loads Phase 1 model (no retraining)
- Runs agreement-based XAI: SHAP primary + LIME + AIX360 validators
- Assigns confidence: HIGH / MEDIUM / LOW per feature per customer
- Generates WHAT + WHY business reasoning
- Produces watchlist with explainable churn reasons

**Design principle:** Confidence comes from inter-method AGREEMENT, not score averaging.

In [ ]:
# Step 1 — Install XAI dependencies
!pip install shap lime --quiet
# AIX360 (optional — system degrades gracefully without it)
# !pip install aix360 --quiet

import shap, lime
print(f'shap: {shap.__version__}')
print(f'lime: {lime.__version__}')
print('Dependencies ready.')

In [ ]:
# Step 2 — Mount Drive (same location as Phase 1)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/CustomerHealthForensics'
XAI_DIR = f'{PROJECT_ROOT}/outputs/xai'

import os
os.makedirs(XAI_DIR, exist_ok=True)
print(f'Phase 1 artifacts: {PROJECT_ROOT}/models')
print(f'XAI outputs:       {XAI_DIR}')

In [ ]:
# Step 3 — Upload Phase 2 source files
# Upload: xai_engine.py, reasoning_layer.py, xai_runner.py
# Also re-upload feature_engineering.py from Phase 1
from google.colab import files
print('Select Phase 2 source files (xai_engine.py, reasoning_layer.py, xai_runner.py)')
print('Also include feature_engineering.py from Phase 1 src/ folder')
uploaded = files.upload()

import sys
for fname, data in uploaded.items():
    dest = f"{PROJECT_ROOT}/src/{fname}"
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved: {dest}')

sys.path.insert(0, f'{PROJECT_ROOT}/src')
print('\nSource files ready.')

In [ ]:
# Step 4 — Smoke test (sample=5000, batch=10)
import sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')
from xai_runner import run_xai
from pathlib import Path

print('Running smoke test (5k rows, 10 watchlist explanations)...')
summary = run_xai(
    data_dir     = Path(f'{PROJECT_ROOT}/data'),
    models_dir   = Path(f'{PROJECT_ROOT}/models'),
    output_dir   = Path(f'{PROJECT_ROOT}/outputs/xai'),
    sample_size  = 5_000,
    max_watchlist = 10,
)
print('\n✓ Smoke test passed.')
print(f"Trust score: {summary['confidence_summary']['trust_score']}")

In [ ]:
# Step 5 — Full run (200 watchlist customers)
import importlib, xai_runner
importlib.reload(xai_runner)
from xai_runner import run_xai
from pathlib import Path

summary = run_xai(
    data_dir      = Path(f'{PROJECT_ROOT}/data'),
    models_dir    = Path(f'{PROJECT_ROOT}/models'),
    output_dir    = Path(f'{PROJECT_ROOT}/outputs/xai'),
    max_watchlist = 200,
)

import json
print(json.dumps(summary, indent=2))

In [ ]:
# Step 6 — Inspect a single customer explanation
import json
from pathlib import Path

watchlist_path = Path(f'{PROJECT_ROOT}/outputs/xai/watchlist_reasoning.json')
with open(watchlist_path) as f:
    watchlist = json.load(f)

# Show first critical customer
customer = watchlist[0]
print(f"Customer:          {customer['customer_id']}")
print(f"Churn probability: {customer['churn_probability']}")
print(f"Risk tier:         {customer['risk_tier']}")
print()
print('WHAT changed:')
for w in customer['reasoning']['what_changed']:
    print(f'  • {w}')
print()
print('WHY at risk:')
why = customer['reasoning']['why']
print(f"  Primary driver:  {why['primary_driver']}")
print(f"  Category:        {why['primary_category']}")
print(f"  Interpretation:  {why['primary_why']}")
print()
print('Recommended action:')
print(f"  → {why['recommended_action']}")

In [ ]:
# Step 7 — Global churn drivers
global_path = Path(f'{PROJECT_ROOT}/outputs/xai/global_reasoning.json')
with open(global_path) as f:
    global_r = json.load(f)

print(f"Churn rate: {global_r['churn_rate']}")
print(f"Method:     {global_r['method']}")
print()
print('WHAT behavioral patterns are driving churn:')
for p in global_r['reasoning']['what_patterns']:
    print(f'  • {p}')
print()
print('WHY explanations:')
for w in global_r['reasoning']['why_explanations']:
    print(f"  [{w['category']}] {w['why']}")
print()
print('Top actions:')
for a in global_r['reasoning']['top_actions']:
    print(f'  → {a}')

In [ ]:
# Step 8 — Confidence distribution
conf_path = Path(f'{PROJECT_ROOT}/outputs/xai/confidence_summary.json')
with open(conf_path) as f:
    conf = json.load(f)

print('Confidence distribution across all explanations:')
print(f"  HIGH:   {conf['HIGH']} features")
print(f"  MEDIUM: {conf['MEDIUM']} features")
print(f"  LOW:    {conf['LOW']} features")
print(f"  Trust score: {conf['trust_score']} (0=all LOW, 1=all HIGH)")
print()
print('Trust score interpretation:')
ts = conf['trust_score']
if ts >= 0.7:
    print('  ✓ HIGH TRUST — explanations are reliable and consistent')
elif ts >= 0.4:
    print('  ⚠ MEDIUM TRUST — most explanations confirmed by validators')
else:
    print('  ✗ LOW TRUST — disagreement between methods, investigate model')

In [ ]:
# Step 9 — List all supported WHAT/WHY questions
from reasoning_layer import list_supported_questions
list_supported_questions()

---
## ✅ Phase 2 Complete

**Artifacts saved to Drive:**
```
outputs/xai/
  global_explanation.json      ← population feature importance
  global_reasoning.json        ← WHAT/WHY globally
  watchlist_explanations.json  ← per-customer XAI (Critical tier)
  watchlist_reasoning.json     ← WHAT/WHY per customer
  confidence_summary.json      ← HIGH/MEDIUM/LOW distribution
  xai_summary.json             ← full Phase 2 metadata
```

**Next:** Phase 3 — Cohort & Segmentation Engine